# MCP Client Testing - Full MCP Protocol

This notebook tests the `search_interaction_events` tool using a real MCP client connection to verify full protocol compliance.

**Prerequisites:** Make sure your MCP server is running on `localhost:8005`

## Setup: Import Dependencies and Check Connection

First, let's import the required MCP client libraries and verify we can connect.

In [21]:
import asyncio
import json
from fastmcp import Client
import httpx

print('✅ MCP client libraries imported successfully')
print('🔌 Ready to test MCP server connection')

✅ MCP client libraries imported successfully
🔌 Ready to test MCP server connection


## Test 1: Basic HTTP Connection Test

Testing basic connectivity to the MCP server.

In [ ]:
async def test_basic_connection():
    """Simple connection test to verify server is responding."""
    print("🔗 Testing basic MCP connection...")
    
    try:
        # Use FastMCP client like the working test_mcp_client.py
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            print("✅ Successfully connected to MCP server")
            
            # Basic connectivity test - list tools
            tools = await client.list_tools()
            print(f"✅ Server responding with {len(tools)} tools")
            
            return True
            
    except Exception as e:
        print(f"❌ Connection test failed: {e}")
        print("   Make sure the server is running on http://localhost:8005")
        return False

# Run the test
connection_result = await test_basic_connection()
if connection_result:
    print('✅ Basic connection test PASSED')
else:
    print('❌ Basic connection test FAILED')

## Test 2: Full MCP Client Session - List Tools

Testing proper MCP client session with tool discovery.

In [23]:
async def test_list_tools():
    """Test MCP server tool listing using FastMCP client."""
    print("📋 Testing MCP tool listing...")
    
    try:
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            print("✅ Connected to MCP server")

            # List available tools
            tools = await client.list_tools()

            print(f"Found {len(tools)} tools:")
            for i, tool in enumerate(tools, 1):
                print(f"  {i}. {tool.name}")
                if tool.description:
                    print(f"     Description: {tool.description[:100]}...")

            # Look for our search tool
            search_tool = next((t for t in tools if t.name == "search_interaction_events"), None)
            if search_tool:
                print("✅ Found search_interaction_events tool!")
                print(f"   Parameters: {search_tool.inputSchema}")
                return True
            else:
                print("❌ search_interaction_events tool not found!")
                return False

    except Exception as e:
        print(f"❌ Tool listing failed: {e}")
        return False

# Run the test
tools_result = await test_list_tools()
if tools_result:
    print('✅ Tool listing test PASSED')
else:
    print('❌ Tool listing test FAILED')

📋 Testing MCP tool listing...
✅ Connected to MCP server
Found 3 tools:
  1. add
     Description: Add two numbers and return the sum as {"sum": int}....
  2. get_event_descriptions
     Description: Return a mapping of all event names and their descriptions from the data catalog.
The key is the eve...
  3. search_interaction_events
     Description: Search for interaction events in the data catalog that match the provided query.

This tool searches...
✅ Found search_interaction_events tool!
   Parameters: {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'max_results': {'default': 10, 'title': 'Max Results', 'type': 'integer'}}, 'required': ['query'], 'type': 'object'}
✅ Tool listing test PASSED


## Test 3: Exact Match Search Test

Testing the search tool with exact event name matching.

In [24]:
async def test_exact_match():
    """Test exact match search functionality."""
    print("🎯 Testing exact match search...")
    
    try:
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            result = await client.call_tool(
                "search_interaction_events",
                {"query": "wayfinder_start", "max_results": 5}
            )

            print(f"✅ Exact match search successful!")
            print(f"Result type: {type(result)}")
            
            if hasattr(result, 'content') and result.content:
                content = result.content[0]
                if hasattr(content, 'text'):
                    print(f"Result preview: {str(content.text)[:300]}...")
                else:
                    print(f"Result content: {content}")
            else:
                print(f"Result: {result}")
            
            return True

    except Exception as e:
        print(f"❌ Exact match search failed: {e}")
        return False

# Run the test
exact_result = await test_exact_match()
if exact_result:
    print('✅ Exact match test PASSED')
else:
    print('❌ Exact match test FAILED')

🎯 Testing exact match search...
✅ Exact match search successful!
Result type: <class 'fastmcp.client.client.CallToolResult'>
Result preview: [{"event_name":"wayfinder_start","trigger":"This event should be triggered when a user clicks on one of the first wayfinder questions, located on the homepage.","confidence_score":100.0,"stream":"WS1 - Website","tool_area":"Wayfinder","parameters":["option_selected","attempt"],"requirements_status":...
✅ Exact match test PASSED


## Test 4: Fuzzy Search Test

Testing fuzzy matching with partial event names.

In [25]:
async def test_fuzzy_search():
    """Test fuzzy search functionality."""
    print("🔍 Testing fuzzy search...")
    
    try:
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            result = await client.call_tool(
                "search_interaction_events",
                {"query": "wayfinder", "max_results": 5}
            )

            print(f"✅ Fuzzy search successful!")
            
            if hasattr(result, 'content') and result.content:
                content = result.content[0]
                if hasattr(content, 'text'):
                    response_text = str(content.text)
                    print(f"Response preview: {response_text[:300]}...")
                    
                    # Check for wayfinder-related terms in response
                    if "wayfinder" in response_text.lower():
                        print("✅ Found wayfinder-related results")
                else:
                    print(f"Result content: {content}")
            else:
                print(f"Result: {result}")
            
            return True

    except Exception as e:
        print(f"❌ Fuzzy search failed: {e}")
        return False

# Run the test
fuzzy_result = await test_fuzzy_search()
if fuzzy_result:
    print('✅ Fuzzy search test PASSED')
else:
    print('❌ Fuzzy search test FAILED')

🔍 Testing fuzzy search...
✅ Fuzzy search successful!
Response preview: [{"event_name":"wayfinder_start","trigger":"This event should be triggered when a user clicks on one of the first wayfinder questions, located on the homepage.","confidence_score":100,"stream":"WS1 - Website","tool_area":"Wayfinder","parameters":["option_selected","attempt"],"requirements_status":"R...
✅ Found wayfinder-related results
✅ Fuzzy search test PASSED


## Test 5: Keyword Search Test

Testing keyword search in event triggers and descriptions.

In [26]:
async def test_keyword_search():
    """Test keyword search functionality."""
    print("🔑 Testing keyword search...")
    
    try:
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            result = await client.call_tool(
                "search_interaction_events",
                {"query": "user clicks button", "max_results": 3}
            )

            print(f"✅ Keyword search successful!")
            
            if hasattr(result, 'content') and result.content:
                content = result.content[0]
                if hasattr(content, 'text'):
                    response_text = str(content.text)
                    print(f"Response preview: {response_text[:300]}...")
                    
                    # Check for click-related terms
                    if any(term in response_text.lower() for term in ["click", "button", "user"]):
                        print("✅ Found click-related results")
                else:
                    print(f"Result content: {content}")
            else:
                print(f"Result: {result}")
            
            return True

    except Exception as e:
        print(f"❌ Keyword search failed: {e}")
        return False

# Run the test
keyword_result = await test_keyword_search()
if keyword_result:
    print('✅ Keyword search test PASSED')
else:
    print('❌ Keyword search test FAILED')

🔑 Testing keyword search...
✅ Keyword search successful!
Response preview: [{"event_name":"wayfinder_start","trigger":"This event should be triggered when a user clicks on one of the first wayfinder questions, located on the homepage.","confidence_score":78,"stream":"WS1 - Website","tool_area":"Wayfinder","parameters":["option_selected","attempt"],"requirements_status":"Re...
✅ Found click-related results
✅ Keyword search test PASSED


## Test 6: Edge Case Testing

Testing edge cases like empty queries and result limits.

In [27]:
async def test_edge_cases():
    """Test edge cases and error handling."""
    print("🧪 Testing edge cases...")
    
    try:
        client = Client("http://localhost:8005/mcp")
        
        async with client:
            # Test empty query
            print("\n1. Testing empty query...")
            try:
                result = await client.call_tool(
                    "search_interaction_events",
                    {"query": "", "max_results": 5}
                )
                print("✅ Empty query handled gracefully")
            except Exception as e:
                print(f"❌ Empty query failed: {e}")
                return False

            # Test non-matching query
            print("\n2. Testing non-matching query...")
            try:
                result = await client.call_tool(
                    "search_interaction_events",
                    {"query": "xyz123nonexistent", "max_results": 5}
                )
                print("✅ Non-matching query handled gracefully")
            except Exception as e:
                print(f"❌ Non-matching query failed: {e}")
                return False

            # Test result limit
            print("\n3. Testing result limit...")
            try:
                result = await client.call_tool(
                    "search_interaction_events",
                    {"query": "interaction", "max_results": 2}
                )
                print("✅ Result limit handled correctly")
                return True
            except Exception as e:
                print(f"❌ Result limit test failed: {e}")
                return False

    except Exception as e:
        print(f"❌ Edge case testing failed: {e}")
        return False

# Run the test
edge_result = await test_edge_cases()
if edge_result:
    print('✅ Edge case testing PASSED')
else:
    print('❌ Edge case testing FAILED')

🧪 Testing edge cases...

1. Testing empty query...
✅ Empty query handled gracefully

2. Testing non-matching query...
✅ Non-matching query handled gracefully

3. Testing result limit...
✅ Result limit handled correctly
✅ Edge case testing PASSED


## Final MCP Client Test Summary

Overall assessment of MCP client functionality.

In [28]:
print('🏁 MCP CLIENT TESTING COMPLETE')
print('=' * 60)

# Collect all test results
test_results = [
    ("Basic Connection", connection_result if 'connection_result' in locals() else False),
    ("Tool Listing", tools_result if 'tools_result' in locals() else False),
    ("Exact Match Search", exact_result if 'exact_result' in locals() else False),
    ("Fuzzy Search", fuzzy_result if 'fuzzy_result' in locals() else False),
    ("Keyword Search", keyword_result if 'keyword_result' in locals() else False),
    ("Edge Cases", edge_result if 'edge_result' in locals() else False)
]

# Display results
passed_tests = 0
for test_name, result in test_results:
    status = "✅ PASSED" if result else "❌ FAILED"
    print(f"{test_name:.<25} {status}")
    if result:
        passed_tests += 1

print(f"\nOverall: {passed_tests}/{len(test_results)} tests passed")

if passed_tests == len(test_results):
    print('\n🎉 ALL MCP CLIENT TESTS PASSED!')
    print('✅ MCP server is fully functional')
    print('✅ search_interaction_events tool works via MCP protocol')
    print('✅ Ready for production deployment!')
    print('📝 QA APPROVED - Ready for documentation!')
elif passed_tests >= 4:
    print('\n⚠️  Most tests passed - minor issues detected')
    print('✅ Core functionality is working')
    print('⚠️  Review failed tests before production')
else:
    print('\n❌ Major issues detected')
    print('🔧 Fix failing tests before proceeding')

print('=' * 60)

🏁 MCP CLIENT TESTING COMPLETE
Basic Connection......... ❌ FAILED
Tool Listing............. ✅ PASSED
Exact Match Search....... ✅ PASSED
Fuzzy Search............. ✅ PASSED
Keyword Search........... ✅ PASSED
Edge Cases............... ✅ PASSED

Overall: 5/6 tests passed

⚠️  Most tests passed - minor issues detected
✅ Core functionality is working
⚠️  Review failed tests before production
